# 02 Train Fraud Model

Phase 4a: gradient-boosted transaction fraud classifier for Emberline Studios. Trains only on purchases at least 75 days before as_of_date (2026-12-31) so chargeback labels are mature, uses a time-based split, tracks everything in MLflow, registers `fraud_txn_classifier@champion` in Unity Catalog, and scores every transaction into `ml_fraud_txn_scores`.

In [0]:
from pyspark.sql import functions as F, Window

CATALOG = "mg_aws_serverless_workspace_catalog"
SCHEMA = "game_cx_players"
FQ = f"{CATALOG}.{SCHEMA}"
AS_OF_DATE = "2026-12-31"
MATURE_CUTOFF = "2026-10-17"   # AS_OF_DATE - 75 days: chargeback labels final on or before this date
TEST_START = "2026-06-01"      # time-based split: train < TEST_START <= test <= MATURE_CUTOFF
REVIEW_CAPACITY = 0.01         # analysts review the top 1% of transactions by score
EXPERIMENT = "/Users/mayur.gupta@databricks.com/game-cx-players-fraud"
MODEL_NAME = f"{FQ}.fraud_txn_classifier"
GT = f"/Volumes/{CATALOG}/{SCHEMA}/ground_truth"
print(FQ, MATURE_CUTOFF, TEST_START)

In [0]:
def build_fraud_features():
    """Build one feature row per transaction from silver tables.

    Features: account age, billing vs IP country mismatch, emulator / rooted device,
    accounts sharing the device fingerprint, 24h and 7d purchase velocity, amount,
    payment method, item type and hour of day. Label: the transaction has a chargeback.

    Returns:
        Spark DataFrame keyed by txn_id.
    """
    txn = spark.table(f"{FQ}.silver_transactions")
    players = spark.table(f"{FQ}.silver_players").select("player_id", "registration_date")
    devices = spark.table(f"{FQ}.silver_devices")
    fp_accounts = devices.groupBy("device_fingerprint").agg(F.countDistinct("player_id").alias("shared_fp_accounts"))
    dev = (devices.join(fp_accounts, "device_fingerprint")
           .select("device_id", F.col("is_emulator").cast("int").alias("is_emulator"),
                   F.col("is_rooted_or_jailbroken").cast("int").alias("is_rooted"), "shared_fp_accounts"))
    cb = spark.table(f"{FQ}.silver_chargebacks").groupBy("txn_id").agg(F.sum("disputed_amount_usd").alias("disputed_usd"))

    ts = F.col("txn_ts").cast("timestamp").cast("long")
    w = Window.partitionBy("player_id").orderBy(ts)
    return (txn.join(players, "player_id", "left").join(dev, "device_id", "left").join(cb, "txn_id", "left")
            .withColumn("account_age_days", F.datediff("txn_date", "registration_date"))
            .withColumn("country_mismatch", (F.col("billing_country") != F.col("ip_country")).cast("int"))
            .withColumn("txns_prior_24h", F.count("*").over(w.rangeBetween(-86400, -1)))
            .withColumn("txns_prior_7d", F.count("*").over(w.rangeBetween(-7 * 86400, -1)))
            .withColumn("txn_hour", F.hour(F.col("txn_ts").cast("timestamp")))
            .withColumn("label", F.col("disputed_usd").isNotNull().cast("int"))
            .select("txn_id", "player_id", "txn_date", "amount_usd", "payment_method", "item_type",
                    "account_age_days", "country_mismatch", "is_emulator", "is_rooted", "shared_fp_accounts",
                    "txns_prior_24h", "txns_prior_7d", "txn_hour", F.coalesce("disputed_usd", F.lit(0.0)).alias("disputed_usd"), "label"))

feat = build_fraud_features()
feat.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.ml_feat_fraud_txn")
feat = spark.table(f"{FQ}.ml_feat_fraud_txn")
display(feat.agg(F.count("*").alias("rows"), F.sum("label").alias("chargebacks"),
                 F.sum(F.when(F.col("txn_date") <= MATURE_CUTOFF, 1)).alias("mature_rows"),
                 F.sum(F.when(F.col("is_emulator").isNull(), 1)).alias("rows_missing_device")))

In [0]:
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Features are built in PySpark (948K rows, windowed velocity). The ~850K mature rows x 12 columns fit in
# memory, so training uses pandas + scikit-learn: faster to iterate than Spark ML at this size.
NUM = ["amount_usd", "account_age_days", "country_mismatch", "is_emulator", "is_rooted",
       "shared_fp_accounts", "txns_prior_24h", "txns_prior_7d", "txn_hour"]
CAT = ["payment_method", "item_type"]
mature = feat.filter(F.col("txn_date") <= MATURE_CUTOFF).toPandas()
train = mature[mature.txn_date < pd.Timestamp(TEST_START).date()]
test = mature[mature.txn_date >= pd.Timestamp(TEST_START).date()]


def make_pipeline():
    """Return a scikit-learn pipeline: one-hot categoricals + histogram gradient boosting."""
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough")
    gbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
                                         class_weight="balanced", random_state=42)
    return Pipeline([("pre", pre), ("gbm", gbm)])


def review_metrics(y, score, disputed, capacity):
    """Precision, recall and chargeback dollars caught when reviewing the top `capacity` share by score."""
    k = max(1, int(len(score) * capacity))
    top = np.argsort(-score)[:k]
    return {"precision_at_capacity": float(y[top].mean()),
            "recall_at_capacity": float(y[top].sum() / max(1, y.sum())),
            "chargeback_usd_caught": float(disputed[top].sum()),
            "chargeback_usd_caught_pct": float(disputed[top].sum() / max(1e-9, disputed.sum())),
            "score_threshold": float(score[top].min())}


mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name="hgb_fraud_txn") as run:
    model = make_pipeline().fit(train[CAT + NUM], train.label)
    p = model.predict_proba(test[CAT + NUM])[:, 1]
    y, d = test.label.to_numpy(), test.disputed_usd.to_numpy()
    metrics = {"pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
               "test_base_rate": float(y.mean()), **review_metrics(y, p, d, REVIEW_CAPACITY)}
    mlflow.log_params({"train_rows": len(train), "test_rows": len(test), "mature_cutoff": MATURE_CUTOFF,
                       "test_start": TEST_START, "review_capacity": REVIEW_CAPACITY, "features": ",".join(CAT + NUM)})
    mlflow.log_metrics(metrics)
    sig = infer_signature(test[CAT + NUM].head(100), p[:100])
    mlflow.sklearn.log_model(model, name="model", signature=sig, input_example=test[CAT + NUM].head(5),
                             registered_model_name=MODEL_NAME)
    RUN_ID = run.info.run_id
THRESHOLD = metrics["score_threshold"]
print(RUN_ID)
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

In [0]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'") if v.run_id == RUN_ID)
client.set_registered_model_alias(MODEL_NAME, "champion", version)
print(f"{MODEL_NAME} v{version} -> @champion")

In [0]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
all_pdf = feat.select("txn_id", "player_id", "txn_date", *CAT, *NUM).toPandas()
all_pdf["fraud_score"] = champion.predict_proba(all_pdf[CAT + NUM])[:, 1]
scores = (spark.createDataFrame(all_pdf[["txn_id", "player_id", "txn_date", "fraud_score"]])
          .withColumn("is_flagged", F.col("fraud_score") >= F.lit(THRESHOLD))
          .withColumn("model_version", F.lit(int(version)))
          .withColumn("scored_as_of", F.lit(AS_OF_DATE).cast("date")))
(scores.write.mode("overwrite").option("overwriteSchema", "true")
       .option("description", "Fraud score per transaction from fraud_txn_classifier@champion. is_flagged = score at or above the top-1% review threshold from the test window.")
       .saveAsTable(f"{FQ}.ml_fraud_txn_scores"))
display(spark.sql(f"""
SELECT
  COUNT(*) AS scored_rows,
  COUNT(DISTINCT txn_id) AS distinct_txns,
  SUM(CAST(is_flagged AS INT)) AS flagged,
  ROUND(AVG(fraud_score), 4) AS avg_score
FROM {FQ}.ml_fraud_txn_scores
"""))

In [0]:
gt_players = spark.read.parquet(f"{GT}/gt_players").select("player_id", "is_fraud_ring")
player_flag = (spark.table(f"{FQ}.ml_fraud_txn_scores").groupBy("player_id")
               .agg(F.max(F.col("is_flagged").cast("int")).alias("flagged")))
ring = gt_players.join(player_flag, "player_id", "left").fillna({"flagged": 0})
ring_stats = ring.groupBy("is_fraud_ring").agg(F.count("*").alias("players"), F.sum("flagged").alias("flagged_players"))
display(ring_stats.withColumn("flag_rate", F.round(F.col("flagged_players") / F.col("players"), 4)))
rr = ring.filter("is_fraud_ring").agg(F.avg("flagged")).first()[0]
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("ring_player_recall", float(rr))
print(f"Ring player recall (any transaction flagged): {rr:.3f}")